In [2]:
import pandas as pd

# 1. 讀取下載好的半年報歷史資料
df_raw = pd.read_csv("US_PortCalls_S.csv")

# 2. 篩選核心維度：只要「貨櫃船 (Container ships)」
# 因為塞港和運價飆漲最核心的主角就是貨櫃船
df_container = df_raw[df_raw['CommercialMarket Label'] == 'Container ships'].copy()

# 3. 篩選對比對象：只要「全球平均 (World)」和「美國 (United States of America)」
# 這可以完美呼應你們目標一想要對比美西與全球的構想
target_economies = ['World', 'United States of America']
df_filtered = df_container[df_container['Economy Label'].isin(target_economies)].copy()

# 4. 只保留網頁與圖表需要的「精簡欄位」，讓資料變乾淨
# 移除那些充滿 Missing value 和 Footnote 的混亂欄位
keep_columns = [
    'Period Label',                               # 時間軸 (如 S1 2018)
    'Economy Label',                              # 國家地區
    'Median time in port (days)',                 # 塞港指標：在港天數中位數
    'Average container carrying capacity (TEU) per container ship' # 規模指標：船隻平均載箱量
]
df_clean = df_filtered[keep_columns]

# 5. 依據時間順序重新排序
df_clean = df_clean.sort_values(by=['Economy Label', 'Period Label']).reset_index(drop=True)

# 6. 查看清洗完的成果
print("=== 清洗完成的乾淨資料前 5 筆 ===")
print(df_clean.head())

# 7. 匯出成一個新的 CSV 檔，直接給後面做互動網頁（目標四）使用
df_clean.to_csv("clean_port_performance.csv", index=False)
print("\n🎉 成功匯出 clean_port_performance.csv！第一步順利完成！")

=== 清洗完成的乾淨資料前 5 筆 ===
  Period Label             Economy Label  Median time in port (days)  \
0    S1   2018  United States of America                      0.9800   
1    S1   2019  United States of America                      1.0358   
2    S1   2020  United States of America                      1.0132   
3    S1   2021  United States of America                      1.2069   
4    S1   2022  United States of America                      1.4326   

   Average container carrying capacity (TEU) per container ship  
0                                             5254.0             
1                                             5364.0             
2                                             5237.0             
3                                             5503.0             
4                                             5183.0             

🎉 成功匯出 clean_port_performance.csv！第一步順利完成！


In [3]:
!pip install seaborn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 3.6 MB/s eta 0:00:03
   ------- -------------------------------- 1.6/8.2 MB 4.1 MB/s eta 0:00:02
   ---------- ----------------------------- 2.1/8.2 MB 3.6 MB/s eta 0:00:02
   ------------ --------------------------- 2.6/8.2 MB 3.4 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.2 MB 3.2 MB/s eta 0:00:02
   ----------------- ---------------------- 3.7/8.2 MB 2.9 MB/s eta 0:00:02
   ------------------- -------------------- 3.9/8.2 MB 2.8 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.2 MB 2.6 MB/s eta 0:00:02
   ---------------------- ----------------- 4.7/8.2 MB 2.5 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.2 MB 2.5 MB/s eta 0:00:02
   -------------------------- ------------- 5.5/8.2 MB 2.3 MB/s eta 0:00:02
   ---------------------------- ----------- 5.8/8.2 MB 2.3 MB/s eta 0:00:02
   ----------------

ERROR: Could not install packages due to an OSError: [WinError 32] 程序無法存取檔案，因為檔案正由另一個程序使用。: 'C:\\Users\\User\\.conda\\envs\\py313\\Lib\\site-packages\\pandas\\_config\\__init__.py'
Consider using the `--user` option or check the permissions.



  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached numpy-2.4.6-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl (226 kB)
Using cached cycler-0.12.1-py3-none

In [20]:
import subprocess
import time
import os
import re

# =====================================================================
# 1. 建立並寫入 app.py 網頁檔案
# =====================================================================
code_content = """
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="全球港口績效儀表板", layout="wide")
st.title("🚢 全球港口績效與塞港監測互動儀表板")
st.subheader("專案實作：跨時期港口停泊時間對比")

@st.cache_data
def load_and_clean_data():
    raw_df = pd.read_csv("US_PortCalls_S.csv", na_values=["Not available or not separately reported"])
    df_container = raw_df[raw_df['CommercialMarket Label'] == 'Container ships'].copy()
    df_container['Median time in port (days)'] = pd.to_numeric(df_container['Median time in port (days)'], errors='coerce')
    clean_df = df_container.dropna(subset=['Median time in port (days)', 'Period Label', 'Economy Label'])
    return clean_df

df = load_and_clean_data()

st.sidebar.header("📊 網頁控制面板")
all_economies = df['Economy Label'].unique().tolist()
selected_economies = st.sidebar.multiselect(
    "選擇想要對比的國家/地區：",
    options=all_economies,
    default=all_economies
)

df_filtered = df[df['Economy Label'].isin(selected_economies)]

fig = px.line(
    df_filtered, 
    x='Period Label', 
    y='Median time in port (days)', 
    color='Economy Label',
    markers=True,
    title="2018-2023 貨櫃船在港時間中位數（天）趨勢對比",
    labels={'Median time in port (days)': '在港時間 (天)', 'Period Label': '時間軸 (半年報)'}
)
fig.update_layout(hovermode="x unified", xtickangle=45)
st.plotly_chart(fig, use_container_width=True)
st.info("💡 互動提示：你可以用滑鼠游標移到圖表的線條上，會動態顯示精確的數值；也可以利用左側控制面板自由隱藏/顯示特定國家。")
"""

with open("app.py", "w", encoding="utf-8") as f:
    f.write(code_content)

print("🎉 1. app.py 網頁主程式寫入成功！")

# =====================================================================
# 2. 徹底重置與清理環境
# =====================================================================
os.system("fuser -k 8501/tcp > /dev/null 2>&1")
os.system("pkill -f ngrok > /dev/null 2>&1")
time.sleep(1)

print("⏳ 2. 正在背景啟動 Streamlit 服務...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"], 
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)

# =====================================================================
# 3. 下載並建立 ngrok 通道
# =====================================================================
print("🚀 3. 正在開啟安全對外通道（已加入延遲偵測防護）...")
if not os.path.exists("/usr/local/bin/ngrok") and os.system("which ngrok > /dev/null 2>&1") != 0:
    os.system("curl -s https://bin.equinox.io/c/bO48Sg3MVO7/ngrok-v3-stable-linux-amd64.tgz | tar -xz -C /usr/local/bin")

# 移除舊的 log 檔案
if os.path.exists("ngrok.log"):
    try:
        os.remove("ngrok.log")
    except:
        pass

# 啟動 ngrok 並將輸出導向到日誌
with open("ngrok.log", "w") as log_file:
    subprocess.Popen(["ngrok", "http", "8501"], stdout=log_file, stderr=log_file)

# =====================================================================
# 4. 智慧重試機制：確保 log 檔案存在並包含網址後才進行抓取
# =====================================================================
public_url = None
print("⏳ 正在等待網路分配公開網址...")

for i in range(10): # 最多等待 10 秒
    time.sleep(1)
    if os.path.exists("ngrok.log"):
        with open("ngrok.log", "r") as f:
            log_content = f.read()
            # 偵測日誌裡有沒有吐出 ngrok-free.app 的網址
            match = re.search(r"url=(https://[a-zA-Z0-9\-]+\.ngrok-free\.app)", log_content)
            if match:
                public_url = match.group(1)
                break # 抓到了就立刻跳出迴圈，不用繼續等

# =====================================================================
# 5. 印出最終輸出結果
# =====================================================================
print("\n" + "="*60)
if public_url:
    print("🎉 網頁對外通道已完美打通！")
    print("👉 請點擊下方這個專屬網址打開你的儀表板：")
    print(f"🔗 {public_url}")
    print("="*60 + "\n")
    print("💡 提示：進去後如果看到 ngrok 的警告頁面，直接點選藍色的 [Visit Site] 即可進入網頁！")
else:
    print("❌ 網路通道建立超時。")
    print("💡 解決辦法：請直接重新執行一次這一格即可！")
print("="*60)

🎉 1. app.py 網頁主程式寫入成功！
⏳ 2. 正在背景啟動 Streamlit 服務...
🚀 3. 正在開啟安全對外通道（已加入延遲偵測防護）...
⏳ 正在等待網路分配公開網址...

❌ 網路通道建立超時。
💡 解決辦法：請直接重新執行一次這一格即可！


In [21]:
import os
import time
import subprocess

# 1. 強制殺掉以前所有殘留卡住的後台（Streamlit 和所有穿透工具）
os.system("fuser -k 8501/tcp > /dev/null 2>&1")
os.system("pkill -f ngrok > /dev/null 2>&1")
os.system("pkill -f localtunnel > /dev/null 2>&1")
os.system("pkill -f lt > /dev/null 2>&1")
time.sleep(2)

print("🧹 背景環境已清理乾淨！")

# 2. 重新在背景啟動 Streamlit
print("⏳ 正在啟動網頁服務...")
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"], 
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 3. 獲取本機 IP 位址 (這等一下要在網頁輸入當密碼，請先拿手機拍下這串數字)
print("\n" + "="*50)
print("🔑 你的 Tunnel 登入密碼（Endpoint IP）為：")
os.system("curl -s ipv4.icanhazip.com")
print("="*50 + "\n")

# 4. 安裝並直接用 localtunnel 撈出網址
print("🚀 正在生成對外網址...")
os.system("npm install -g localtunnel > /dev/null 2>&1")

# 啟動 localtunnel 並直接把網址印在畫面上
os.system("npx localtunnel --port 8501")

🧹 背景環境已清理乾淨！
⏳ 正在啟動網頁服務...

🔑 你的 Tunnel 登入密碼（Endpoint IP）為：

🚀 正在生成對外網址...


1